# LATTE: SWOT frontal cyclonic eddy

Single snapshot: 1 November 2023, cycle 005/pass 545. Approximate centre 151.75°E, 34.75°S.
Use the supplied **absolute geostrophic velocities**, not total surface velocity or velocity anomalies.
Edit `config.json`, restart the kernel, and run all cells. Paths are relative to this project.

The product velocities are derived from filtered SSH using spline fitting plus mean dynamic currents.
The product warns that denoising may affect wavelengths below 50 km. The core is near the swath gap;
fits may be influenced by missing coverage and the neighbouring front. Selection radii are provisional.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'functions.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT))
from functions import (refine_latte, out_core_param_fit, model_uv_at_xy,
                       vector_R2, axis_ratio_from_Q, plot_outer_core_fit)
config = json.loads((ROOT / 'config.json').read_text())
display(config)


{'input_file': 'data/cycle_005/SWOT_L3_LR_SSH_Expert_005_532_20231031T155229_20231031T164355_v3.0.nc',
 'fields': {'longitude': 'longitude',
  'latitude': 'latitude',
  'time': 'time',
  'height': 'ssha_filtered',
  'u': 'ugos_filtered',
  'v': 'vgos_filtered',
  'quality': 'quality_flag'},
 'coordinates': {'origin_longitude_deg': 151.75,
  'origin_latitude_deg': -34.75,
  'metres_per_degree': 111000.0},
 'selection': {'quality_values': [0],
  'extent_km': [-70, 70, -65, 65],
  'pixel_stride': 1},
 'latte': {'initial_center_m': [0, 0],
  'initial_Q': [[1, 0], [0, 1]],
  'inner_radius_m': 15000,
  'outer_radius_m': 50000},
 'outer_fit': {'Rc0': None, 'Rc_max': None, 'maxfev': 10000},
 'sensitivity': {'inner_radii_m': [10000, 15000, 20000, 25000, 30000, 35000],
  'outer_radii_m': [30000, 40000, 50000, 60000, 70000, 80000]},
 'plot': {'quiver_stride': 5,
  'quiver_scale': 20,
  'figure_size': [15, 5],
  'radial_curve_points': 400,
  'cmap': 'Blues_r',
  'scatter_size': 20}}

## Load and select observations

Retain only configured quality codes and finite paired velocities. No gap filling or additional
smoothing. The local approximation is x = Δlongitude × 111000 × cos(origin latitude),
y = Δlatitude × 111000, in metres, with east/north velocity components in m/s.
The scale is configurable; no rotation into along/cross-track directions is applied.
Pixel stride is applied before selection, and is 1 by default (all native samples).


In [2]:
f = config['fields']
c = config['coordinates']
s = config['selection']
stride = s['pixel_stride']
if not isinstance(stride, int) or stride < 1:
    raise ValueError('pixel_stride must be a positive integer')
with xr.open_dataset(ROOT / config['input_file']) as raw:
    provenance = dict(raw.attrs)
    ds = raw[list(dict.fromkeys(f.values()))].isel(
        num_lines=slice(None, None, stride), num_pixels=slice(None, None, stride)).load()
for key in ['u', 'v']:
    if ds[f[key]].attrs.get('units') not in ['m/s', 'm s-1']:
        raise ValueError('Expected velocities in m/s')
if ds[f['longitude']].attrs.get('units') != 'degrees_east' or ds[f['latitude']].attrs.get('units') != 'degrees_north':
    raise ValueError('Expected longitude/latitude in degrees')
lon, lat = ds[f['longitude']].values, ds[f['latitude']].values
x = (lon-c['origin_longitude_deg'])*c['metres_per_degree']*np.cos(np.deg2rad(c['origin_latitude_deg']))
y = (lat-c['origin_latitude_deg'])*c['metres_per_degree']
u, v = ds[f['u']].values, ds[f['v']].values
xmin, xmax, ymin, ymax = np.asarray(s['extent_km'])*1000
region = (x >= xmin)&(x <= xmax)&(y >= ymin)&(y <= ymax)
quality = np.isin(ds[f['quality']].values, s['quality_values'])
valid = region & quality & np.isfinite(x)&np.isfinite(y)&np.isfinite(u)&np.isfinite(v)
xi, yi, ui, vi = [a[valid] for a in [x, y, u, v]]
if not xi.size:
    raise ValueError('No valid vectors in selection')
local_times = ds[f['time']].values[np.any(valid, axis=1)]
display(pd.Series({'product':provenance.get('title'), 'version':provenance.get('product_version'),
                   'doi':provenance.get('doi'), 'first_local_time_UTC':local_times.min(),
                   'last_local_time_UTC':local_times.max(), 'valid_vectors':xi.size,
                   'quality_meanings':ds[f['quality']].attrs.get('flag_meanings'),
                   'velocity_definition':ds[f['u']].attrs.get('comment')}))


ValueError: No valid vectors in selection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for ax, field, label in zip(axes, [ds[f['height']].values, np.hypot(u,v)],
                             ['Filtered SSH anomaly (m)', 'Absolute geostrophic speed (m/s)']):
    mask = region & quality & np.isfinite(field)
    h = ax.scatter(x[mask]/1000, y[mask]/1000, c=field[mask], s=config['plot']['scatter_size'], cmap=config['plot']['cmap'])
    fig.colorbar(h, ax=ax, label=label)
    ax.scatter(*np.asarray(config['latte']['initial_center_m'])/1000, marker='x', c='red')
    ax.set(xlabel='East (km)', ylabel='North (km)', aspect='equal',
           xlim=s['extent_km'][:2], ylim=s['extent_km'][2:])
plt.show()


## Fit the inner geometry, then the radial profile

`refine_latte` selects an initial circular/elliptical core and refines it once using the first fit.
The outer selection is an elliptical-radius limit about the fitted centre. As in the archived
application, the radial fit includes retained inner observations and selects tangential velocities
with the fitted rotation sign. Diagnostics distinguish outer candidates from retained fit samples.
No background flow is subtracted. The centre is free to move; cyclonic rotation is checked, not forced.


In [ ]:
def fit_snapshot(inner_radius, outer_radius):
    if not (0 < inner_radius < outer_radius):
        raise ValueError('Require 0 < inner_radius < outer_radius')
    opts = config['latte']
    inner = {}
    result = refine_latte(xi, yi, ui, vi, center=tuple(opts['initial_center_m']),
                          Q_pre=np.asarray(opts['initial_Q'], float),
                          rho_core=inner_radius, fit_details=inner)
    xc, yc, w_core, Q, omega_core, inner_r2 = result
    record = dict(inner_radius_km=inner_radius/1000, outer_radius_km=outer_radius/1000,
                  inner_status=inner['status'], outer_status='not_attempted',
                  xc_m=xc, yc_m=yc, inner_vorticity_s_1=w_core, inner_R2=inner_r2)
    outer = {}
    outer_mask = np.zeros(xi.size, dtype=bool)
    if inner['status'] == 'optimized':
        dx, dy = xi-xc, yi-yc
        rho2 = Q[0,0]*dx**2+2*Q[0,1]*dx*dy+Q[1,1]*dy**2
        outer_mask = rho2 <= outer_radius**2
        Rc, psi0, omega = out_core_param_fit(
            xi[outer_mask], yi[outer_mask], ui[outer_mask], vi[outer_mask], xc, yc, Q,
            Omega0=omega_core, core_mask=inner['refined_mask'][outer_mask],
            fit_details=outer, **config['outer_fit'])
        uh, vh = model_uv_at_xy(xi[outer_mask], yi[outer_mask], xc, yc, Q, omega, Rc)
        record.update(outer_status=outer['status'], Rc_km=Rc/1000, psi0_m2_s=psi0,
                      Omega_s_1=omega, axis_ratio=axis_ratio_from_Q(Q),
                      centre_longitude=c['origin_longitude_deg']+xc/(c['metres_per_degree']*np.cos(np.deg2rad(c['origin_latitude_deg']))),
                      centre_latitude=c['origin_latitude_deg']+yc/c['metres_per_degree'],
                      cyclonic=bool(omega*np.sin(np.deg2rad(c['origin_latitude_deg'])) > 0) if np.isfinite(omega) else None,
                      inner_samples=int(inner['refined_mask'].sum()), outer_candidates=int(outer_mask.sum()),
                      radial_fit_samples=len(outer.get('observation_indices', [])),
                      vector_R2=vector_R2(ui[outer_mask], vi[outer_mask], uh, vh),
                      vector_RMSE_m_s=float(np.sqrt(np.mean((ui[outer_mask]-uh)**2+(vi[outer_mask]-vh)**2))))
    return record, Q, inner, outer, outer_mask

record, Q, inner, outer, outer_mask = fit_snapshot(
    config['latte']['inner_radius_m'], config['latte']['outer_radius_m'])
display(pd.Series(record))


## Observations, reconstruction and radial diagnostic

Maps use identical speed colours and vector scales. Reconstruction is shown only at valid observed
locations; it does not fill or validate the swath gap. Vector R² and RMSE include **all** outer-region
candidates, including velocities rejected by the radial sign selection. The radial diagnostic alone
uses the retained sign-selected observations. No independence-based uncertainty claim is made.


In [ ]:
if outer.get('status') == 'optimized':
    xc, yc = record['xc_m'], record['yc_m']
    uh, vh = model_uv_at_xy(xi, yi, xc, yc, Q, record['Omega_s_1'], record['Rc_km']*1000)
    fig, axes = plt.subplots(1, 3, figsize=config['plot']['figure_size'], constrained_layout=True)
    vmax = max(np.hypot(ui,vi).max(), np.hypot(uh,vh).max())
    step = config['plot']['quiver_stride']
    if not isinstance(step, int) or step < 1:
        raise ValueError('quiver_stride must be a positive integer')
    theta = np.linspace(0, 2*np.pi, 200)
    eigenvalues, eigenvectors = np.linalg.eigh(Q)
    for ax, U, V, title in zip(axes[:2], [ui,uh], [vi,vh], ['Observed geostrophic flow','LATTE reconstruction']):
        h = ax.scatter(xi/1000, yi/1000, c=np.hypot(U,V), s=config['plot']['scatter_size'], vmin=0, vmax=vmax, cmap=config['plot']['cmap'])
        ax.quiver(xi[::step]/1000, yi[::step]/1000, U[::step], V[::step], scale=config['plot']['quiver_scale'])
        ax.scatter(xc/1000, yc/1000, marker='x', c='red')
        for radius, style in [(config['latte']['inner_radius_m'], ':'), (config['latte']['outer_radius_m'], '--')]:
            ellipse = eigenvectors @ (radius/np.sqrt(eigenvalues)[:,None]*np.array([np.cos(theta),np.sin(theta)]))
            ax.plot((ellipse[0]+xc)/1000, (ellipse[1]+yc)/1000, style, color='red')
        ax.set(title=title, xlabel='East (km)', ylabel='North (km)', aspect='equal',
               xlim=s['extent_km'][:2], ylim=s['extent_km'][2:])
    fig.colorbar(h, ax=list(axes[:2]), label='Speed (m/s)')
    plot_outer_core_fit(outer, ax=axes[2], km_flag=True, ci_flag=False, pred_flag=False, n_curve=config['plot']['radial_curve_points'])
    axes[2].set_title('Radial fit')
    plt.show()
    # residual = np.hypot(ui-uh, vi-vh)
    # fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)
    # h = ax.scatter(xi[outer_mask]/1000, yi[outer_mask]/1000, c=residual[outer_mask], s=config['plot']['scatter_size'], cmap=config['plot']['cmap'])
    # fig.colorbar(h, ax=ax, label='Vector residual magnitude (m/s)')
    # ax.set(xlabel='East (km)', ylabel='North (km)', aspect='equal', title='Residuals within outer selection')
    # plt.show()
else:
    print('Reconstruction unavailable:', record['inner_status'], record['outer_status'])


In [ ]:
from IPython.display import Markdown

if outer.get('status') == 'optimized':
    Rc = record['Rc_km']
    eigenvalues = np.linalg.eigvalsh(Q)
    omega = record['Omega_s_1']
    maximum_speed = abs(omega) * (Rc * 1000 / np.sqrt(2)) * np.exp(-0.5) * np.sqrt(eigenvalues.max())
    # For this Gaussian ESP field, centre vorticity is Omega * trace(Q).
    final_vorticity = omega * np.trace(Q)
    direction = 'clockwise' if omega < 0 else 'anticlockwise'
    rotation_type = 'cyclonic' if record['cyclonic'] else 'anticyclonic'
    display(Markdown(f"""### Eddy summary — provisional fit

- **Centre (longitude, latitude):** {record['centre_longitude']:.4f}°E, {abs(record['centre_latitude']):.4f}°{'S' if record['centre_latitude'] < 0 else 'N'}.
- **Centre (x, y):** ({record['xc_m']/1000:.2f}, {record['yc_m']/1000:.2f}) km east/north of the configured origin.
- **Size:** Gaussian characteristic radius **Rc = {Rc:.2f} km**.
- **Maximum tangential speed:** {maximum_speed:.3f} m/s.
- **Rotation:** {direction}, {rotation_type}.
- **Aspect ratio:** {record['axis_ratio']:.3f}.
- **Angular-velocity Ω:** {omega:.3e} s⁻¹ .
- **Centre vorticity $\omega$:** {final_vorticity:.3e} s⁻¹.
- **Fit diagnostics:** inner R² = {record['inner_R2']:.3f}; full vector R² = {record['vector_R2']:.3f}; vector RMSE = {record['vector_RMSE_m_s']:.3f} m/s within the outer selection.

"""))
else:
    display(Markdown('### Eddy summary\n\nNo valid full reconstruction is available. '
                     f"Inner status: `{record['inner_status']}`; outer status: `{record['outer_status']}`."))


## Selection sensitivity

Trial radii are configurable and are not optimisation targets. Inspect centre, geometry, fit quality,
rotation and failures together. Neighbouring pixels are correlated; sample count is not an independent
sample size. A plausible fit still needs assessment against frontal structure and missing coverage.


In [ ]:
sensitivity = pd.DataFrame([
    fit_snapshot(ri, ro)[0]
    for ri in config['sensitivity']['inner_radii_m']
    for ro in config['sensitivity']['outer_radii_m'] if ri < ro])
display(sensitivity)
